# VirtualiZarr → Icechunk in Cloud Storage (Source Coop)

This notebook creates an Icechunk store on Source Coop containing virtual references to Copernicus Marine Service chlorophyll data. The Icechunk store references the original Copernicus data rather than copying it.

## Key Points
- Virtual references point to Copernicus cloudferro S3/HTTPS URLs
- No data duplication - only metadata stored in Icechunk
- Source Coop provides cloud storage for the Icechunk repository
- Users can access the data from anywhere with proper Copernicus credentials

## I need to patch icechunk

Until Source Coop allows CopyObject, I need to not backup the `repo` file when making icechunk commits. 

Make the edits to clone the icechunk repo so I have a local copy. The make edits to not use backup in icechunk/src/asset.rs. Then rebuild and pip install icechunk from latest wheel.
```
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh
source "$HOME/.cargo/env"
cd /home/jovyan/icechunk/icechunk-python
maturin build --release

find /home/jovyan/icechunk -path "*/target/wheels/*.whl"
```

In [1]:
!python -m pip install -q --force-reinstall \
  /home/jovyan/icechunk/target/wheels/icechunk-2.1.1+patched-cp312-abi3-manylinux_2_39_x86_64.whl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xmip 0.7.2 requires cf_xarray>=0.6.0, which is not installed.
xmip 0.7.2 requires xarrayutils, which is not installed.
xmip 0.7.2 requires xgcm<0.7.0, which is not installed.
numba 0.63.1 requires numpy<2.4,>=1.22, but you have numpy 2.5.1 which is incompatible.


In [1]:
import warnings
import time
from pathlib import Path
import json

import xarray as xr
import icechunk
from obstore.store import from_url
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry

warnings.filterwarnings('ignore', category=UserWarning)
icechunk.__version__

'2.1.1+patched'

## Get Copernicus file URLs

Use `copernicusmarine` to get a list of files without downloading them.

In [3]:
# Get file list for 1997 (example - adjust dates as needed)
!copernicusmarine get \
  --dataset-id cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D \
  --dataset-version 202603 \
  --filter "*1997*.nc" \
  --create-file-list copernicus_files_sc.txt

INFO - 2026-07-24T18:02:39Z - Selected dataset version: "202603"
INFO - 2026-07-24T18:02:39Z - Selected dataset part: "default"
INFO - 2026-07-24T18:02:39Z - Listing files on remote server...
11it [00:05,  1.96it/s]
{
  "number_of_files_to_download": 0,
  "status": "002",
  "message": "The request created a file list and then stopped."
}


In [3]:
# Read and convert URLs
with open('copernicus_files_sc.txt', 'r') as f:
    s3_urls = [line.strip() for line in f if line.strip()]

s3_urls.sort()

COPERNICUS_ENDPOINT = "https://s3.waw3-1.cloudferro.com"
def s3_to_https(s3_url):
    if s3_url.startswith('s3://'):
        return f"{COPERNICUS_ENDPOINT}/{s3_url[5:]}"
    return s3_url

https_urls = [s3_to_https(url) for url in s3_urls]
print(f"Found {len(https_urls)} files")
print(f"First: {https_urls[0]}")
print(f"Last: {https_urls[-1]}")

Found 31 files
First: https://s3.waw3-1.cloudferro.com/mdl-native-16/native/OCEANCOLOUR_GLO_BGC_L3_MY_009_103/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D_202603/2024/07/20240701_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
Last: https://s3.waw3-1.cloudferro.com/mdl-native-16/native/OCEANCOLOUR_GLO_BGC_L3_MY_009_103/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D_202603/2024/07/20240731_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc


## Set up remote file configuration

Configure how to access the Copernicus cloudferro files.

In [4]:
# Create object-store handle for the REMOTE Copernicus files
url_prefix = f"{COPERNICUS_ENDPOINT}/"
store = from_url(url_prefix)
registry = ObjectStoreRegistry({url_prefix: store})
parser = HDFParser()

# Configure virtual chunk container
# This tells Icechunk where the actual data chunks live
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=url_prefix,
        store=icechunk.http_store(),
    )
)

print(f"✓ Remote storage configured for: {url_prefix}")

✓ Remote storage configured for: https://s3.waw3-1.cloudferro.com/


## Set up Source Coop storage

Source Coop provides cloud storage for the Icechunk repository.

**Important**: 
1. Get your Source Coop credentials from the 'View Credentials' link
2. Create a `source-creds.json` file (add to `.gitignore`)
3. Never hard-code credentials in notebooks

In [5]:
# Read Source Coop credentials from JSON file
with open("globcolour-source-creds.json") as f:
    source_creds = json.load(f)

# Source Coop bucket information
# Adjust these to match your Source Coop organization and dataset
# https://source.coop/fish-pace/globcolour/<icechunk-name>
storage = icechunk.s3_storage(
    bucket="fish-pace",
    prefix=(
        "globcolour/"
        "test3"
#        "cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D"
    ),
    region=source_creds["region_name"],
    endpoint_url=source_creds["endpoint_url"],
    force_path_style=True,
    access_key_id=source_creds["aws_access_key_id"],
    secret_access_key=source_creds["aws_secret_access_key"],
    session_token=source_creds["aws_session_token"],
)

print("✓ Source Coop storage configured")

✓ Source Coop storage configured


In [6]:
vds = open_virtual_dataset(
        url=https_urls[0],
        parser=parser,
        registry=registry,
        loadable_variables=['time', 'lat', 'lon', 'latitude', 'longitude'],
        decode_times=True,
    )

In [7]:
# Create store if it is empty
try:
    repo = icechunk.Repository.create(storage, config)
    print("Created new Icechunk repo")
except Exception:
    repo = icechunk.Repository.open(storage, config=config)
    print("Opened existing Icechunk repo")

# Create a session
session = repo.writable_session(branch="main")

Opened existing Icechunk repo


In [8]:
session = repo.writable_session("main")

vds.virtualize.to_icechunk(session.store)
try:
    snapshot_id = session.commit("Test one file")
    print(snapshot_id)
except Exception as e:
    print(str(e))
    print(repr(e))
    raise

ZWCT4KZ682CK2XTVK8X0


In [59]:
# Read in the json file
import json
from pathlib import Path

with open("globcolour-source-creds.json") as f:
    source_creds = json.load(f)
    
# This info you get from your storage
source_bucket = "us-west-2.opendata.source.coop"
source_prefix = "fish-pace/globcolour/test2"
source_region = "us-west-2"

storage = icechunk.s3_storage(
    bucket=source_bucket,
    prefix=source_prefix,
    region=source_creds["region_name"],
    access_key_id=source_creds["aws_access_key_id"],
    secret_access_key=source_creds["aws_secret_access_key"],
    session_token=source_creds["aws_session_token"],
    endpoint_url=source_creds["endpoint_url"],
)

In [60]:
# Read in the json file
import json
from pathlib import Path

with open("source-creds.json") as f:
    source_creds = json.load(f)
    
# This info you get from your storage
source_bucket = "us-west-2.opendata.source.coop"
source_prefix = "fish-pace/pace-oci/inregion/PACE_OCI_L3M_CHL"
source_region = "us-west-2"

storage = icechunk.s3_storage(
    bucket=source_bucket,
    prefix=source_prefix,
    region=source_creds["region_name"],
    access_key_id=source_creds["aws_access_key_id"],
    secret_access_key=source_creds["aws_secret_access_key"],
    session_token=source_creds["aws_session_token"],
    endpoint_url=source_creds["endpoint_url"],

)

In [25]:
# Create store if it is empty
try:
    repo = icechunk.Repository.create(storage, config)
    print("Created new Icechunk repo")
except Exception:
    repo = icechunk.Repository.open(storage, config=config)
    print("Opened existing Icechunk repo")

# Create a session
session = repo.writable_session(branch="main")

  2026-07-24T23:41:43.575127Z DEBUG icechunk::repository: Creating Repository
    at icechunk/src/repository.rs:215
    in icechunk::repository::create

  2026-07-24T23:41:43.752026Z DEBUG icechunk::repository: Opening Repository
    at icechunk/src/repository.rs:362
    in icechunk::repository::open

  2026-07-24T23:41:43.752414Z TRACE icechunk::asset_manager: Fetching repo info without cache
    at icechunk/src/asset_manager.rs:518
    in icechunk::asset_manager::fetch_repo_info
    in icechunk::repository::fetch_spec_version

  2026-07-24T23:41:43.752433Z DEBUG icechunk::asset_manager: Downloading repo info
    at icechunk/src/asset_manager.rs:1719
    in icechunk::asset_manager::fetch_repo_info_from_path
    in icechunk::asset_manager::fetch_repo_info
    in icechunk::repository::fetch_spec_version



Opened existing Icechunk repo


  2026-07-24T23:41:43.844928Z TRACE icechunk::repository: Repository version found, spec_version: 2.0
    at icechunk/src/repository.rs:405
    in icechunk::repository::open

  2026-07-24T23:41:43.845621Z TRACE icechunk::asset_manager: Fetching repo info without cache
    at icechunk/src/asset_manager.rs:518
    in icechunk::asset_manager::fetch_repo_info
    in icechunk::repository::get_status
    in icechunk::repository::writable_session with branch: "main"

  2026-07-24T23:41:43.845638Z DEBUG icechunk::asset_manager: Downloading repo info
    at icechunk/src/asset_manager.rs:1719
    in icechunk::asset_manager::fetch_repo_info_from_path
    in icechunk::asset_manager::fetch_repo_info
    in icechunk::repository::get_status
    in icechunk::repository::writable_session with branch: "main"

  2026-07-24T23:41:43.893899Z TRACE icechunk::asset_manager: Repo info cache wasn't latest, updating from none to etag="dc3de0e0ea5ede6f99f079937d34ad8e"
    at icechunk/src/asset_manager.rs:533
  

## Write files to Icechunk

Process each file: virtualize and write/append to Icechunk.

In [52]:
# Process all files
commit_every = 2  # Commit every N files
total_added = 0
start = time.perf_counter()

for i, url in enumerate(https_urls):
    filename = Path(url).name
    
    print(f"[{i+1}/{len(https_urls)}] Processing {filename}...")
    
    # Open file virtually (no data download)
    vds = open_virtual_dataset(
        url=url,
        parser=parser,
        registry=registry,
        loadable_variables=['time', 'lat', 'lon', 'latitude', 'longitude'],
        decode_times=True,
    )
    
    # First file: create, subsequent: append
    if i == 0:
        vds.virtualize.to_icechunk(session.store)
    else:
        vds.virtualize.to_icechunk(session.store, append_dim="time")
    
    total_added += 1
    
    # Commit periodically
    if total_added % commit_every == 0:
        elapsed = time.perf_counter() - start
        snapshot_id = session.commit(f"Add through file {i + 1}")
        print(f"  Committed {total_added} files in {elapsed:.2f}s. Snapshot: {snapshot_id}")
        session = repo.writable_session("main")
        start = time.perf_counter()

# Final commit if needed
if total_added % commit_every != 0:
    snapshot_id = session.commit(f"Final commit: {total_added} files")
    print(f"Final commit: {snapshot_id}")

print(f"\n✓ Successfully added {total_added} files to Icechunk")

[1/31] Processing 20240701_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc...
[2/31] Processing 20240702_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc...


IcechunkError:   x session error: object store error service error
  | 
  | context:
  |    0: icechunk::asset_manager::update_repo_info_internal
  |            with skip_online_check=false
  |              at icechunk/src/asset_manager.rs:815
  |    1: icechunk::asset_manager::update_repo_info
  |              at icechunk/src/asset_manager.rs:784
  |    2: icechunk::session::commit_inner
  |            with Add through file 2 max_concurrent_nodes=1 rewrite_manifests=false commit_method=NewCommit allow_empty=false
  |              at icechunk/src/session.rs:1657
  | 


## Summary

✓ Created Icechunk repository on Source Coop
✓ Stored virtual references to Copernicus cloudferro data
✓ No data duplication - only metadata in Icechunk

### Next Steps

Users can now access this data from anywhere using:

```python
import icechunk
import xarray as xr

url = "https://data.source.coop/your-org/copernicus-marine/chlorophyll"
storage = icechunk.http_storage(url)

# Provide credentials for virtual chunk access (Copernicus data)
credentials = icechunk.containers_credentials({
    "https://s3.waw3-1.cloudferro.com/": icechunk.http_store()
})

repo = icechunk.Repository.open(
    storage,
    authorize_virtual_chunk_access=credentials,
)
session = repo.readonly_session("main")
ds = xr.open_zarr(session.store, consolidated=False)
```

In [41]:
import os

os.environ["RUST_LOG"] = (
    "icechunk=debug,"
    "icechunk_arrow_object_store=debug,"
    "object_store=debug"
)

import icechunk as ic

In [65]:
session = repo.writable_session("main")

vds.virtualize.to_icechunk(session.store)
try:
    snapshot_id = session.commit("Test one file")
    print(snapshot_id)
except Exception as e:
    print(str(e))
    print(repr(e))
    raise
    

  x session error: object store error service error
  | 
  | context:
  |    0: icechunk::asset_manager::update_repo_info_internal
  |            with skip_online_check=false
  |              at icechunk/src/asset_manager.rs:815
  |    1: icechunk::asset_manager::update_repo_info
  |              at icechunk/src/asset_manager.rs:784
  |    2: icechunk::session::commit_inner
  |            with Test one file max_concurrent_nodes=1 rewrite_manifests=false commit_method=NewCommit allow_empty=false
  |              at icechunk/src/session.rs:1657
  | 

icechunk.IcechunkError(message="  x session error: object store error service error
  | 
  | context:
  |    0: icechunk::asset_manager::update_repo_info_internal
  |            with skip_online_check=false
  |              at icechunk/src/asset_manager.rs:815
  |    1: icechunk::asset_manager::update_repo_info
  |              at icechunk/src/asset_manager.rs:784
  |    2: icechunk::session::commit_inner
  |            with Test one file ma

IcechunkError:   x session error: object store error service error
  | 
  | context:
  |    0: icechunk::asset_manager::update_repo_info_internal
  |            with skip_online_check=false
  |              at icechunk/src/asset_manager.rs:815
  |    1: icechunk::asset_manager::update_repo_info
  |              at icechunk/src/asset_manager.rs:784
  |    2: icechunk::session::commit_inner
  |            with Test one file max_concurrent_nodes=1 rewrite_manifests=false commit_method=NewCommit allow_empty=false
  |              at icechunk/src/session.rs:1657
  | 


In [20]:
import icechunk as ic

storage = ic.s3_storage(
    bucket="fish-pace",
    prefix=(
        "globcolour/"
        "test3"
    ),
    region=source_creds["region_name"],
    endpoint_url=source_creds["endpoint_url"],
    force_path_style=True,
    access_key_id=source_creds["aws_access_key_id"],
    secret_access_key=source_creds["aws_secret_access_key"],
    session_token=source_creds["aws_session_token"],
)

storage_settings = ic.StorageSettings(
    unsafe_use_conditional_update=False,
    unsafe_use_conditional_create=False,
    unsafe_use_metadata=False,
)

config = ic.RepositoryConfig(
    storage=storage_settings,
)

repo = ic.Repository.open(
    storage,
    config=config,
    authorize_virtual_chunk_access={
        "https://s3.waw3-1.cloudferro.com/": ic.credentials.HttpAccess,
    },
)

session = repo.writable_session("main")

vds.virtualize.to_icechunk(session.store)
try:
    snapshot_id = session.commit("Test one file")
    print(snapshot_id)
except Exception as e:
    print(str(e))
    print(repr(e))
    raise

  2026-07-24T23:40:04.085008Z DEBUG icechunk::repository: Opening Repository
    at icechunk/src/repository.rs:362
    in icechunk::repository::open

  2026-07-24T23:40:04.085139Z TRACE icechunk::asset_manager: Fetching repo info without cache
    at icechunk/src/asset_manager.rs:518
    in icechunk::asset_manager::fetch_repo_info
    in icechunk::repository::fetch_spec_version

  2026-07-24T23:40:04.085154Z DEBUG icechunk::asset_manager: Downloading repo info
    at icechunk/src/asset_manager.rs:1719
    in icechunk::asset_manager::fetch_repo_info_from_path
    in icechunk::asset_manager::fetch_repo_info
    in icechunk::repository::fetch_spec_version

  2026-07-24T23:40:04.478136Z TRACE icechunk::repository: Repository version found, spec_version: 2.0
    at icechunk/src/repository.rs:405
    in icechunk::repository::open

  2026-07-24T23:40:04.478440Z TRACE icechunk::asset_manager: Fetching repo info without cache
    at icechunk/src/asset_manager.rs:518
    in icechunk::asset_manag

NameError: name 'vds' is not defined

In [78]:
import json
from pathlib import Path

with open("globcolour-source-creds.json") as f:
    source_creds = json.load(f)

storage = icechunk.s3_storage(
    bucket="us-west-2.opendata.source.coop",
    prefix=(
        "fish-pace/globcolour/"
        "cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D"
    ),
    region=source_creds["region_name"],
    access_key_id=source_creds["aws_access_key_id"],
    secret_access_key=source_creds["aws_secret_access_key"],
    session_token=source_creds["aws_session_token"],
)

In [79]:
# Create or open Icechunk repository
try:
    repo = icechunk.Repository.create(storage, config)
    print("Created new Icechunk repo")
except Exception as e:
    repo = icechunk.Repository.open(storage, config=config)
    print("Opened existing Icechunk repo")

# Create a writable session
session = repo.writable_session(branch="main")
print("✓ Ready to write data")

IcechunkError:   x object store error service error
  | 
  | context:
  |    0: icechunk::refs::fetch_branch
  |            with name="main"
  |              at icechunk/src/refs.rs:458
  |    1: icechunk::refs::fetch_branch_tip_v1
  |            with name="main"
  |              at icechunk/src/refs.rs:479
  |    2: icechunk::repository::fetch_spec_version
  |              at icechunk/src/repository.rs:527
  | 
  |-> object store error service error
  |-> service error
  |-> unhandled error (PermanentRedirect)
  `-> Error { code: "PermanentRedirect", message: "The bucket you are attempting to access must be addressed using the specified endpoint. Please send all future requests to this endpoint.",
      aws_request_id: "DXESWHBS8YV0GH4A", s3_extended_request_id: "3CseCaOhweECn0Pedxx3L3hx5kPnUNs6NzF1Yb9B2pIuqcnLrTKMEVnsynZR+vwRxu5TRz3LiYkzq97Nj9txWep3xzQWdOy6" }


In [72]:
session = repo.writable_session("main")

vds.virtualize.to_icechunk(session.store)
try:
    snapshot_id = session.commit("Test one file")
    print(snapshot_id)
except Exception as e:
    print(str(e))
    print(repr(e))
    raise

  x session error: object store error service error
  | 
  | context:
  |    0: icechunk::asset_manager::update_repo_info_internal
  |            with skip_online_check=false
  |              at icechunk/src/asset_manager.rs:815
  |    1: icechunk::asset_manager::update_repo_info
  |              at icechunk/src/asset_manager.rs:784
  |    2: icechunk::session::commit_inner
  |            with Test one file max_concurrent_nodes=1 rewrite_manifests=false commit_method=NewCommit allow_empty=false
  |              at icechunk/src/session.rs:1657
  | 

icechunk.IcechunkError(message="  x session error: object store error service error
  | 
  | context:
  |    0: icechunk::asset_manager::update_repo_info_internal
  |            with skip_online_check=false
  |              at icechunk/src/asset_manager.rs:815
  |    1: icechunk::asset_manager::update_repo_info
  |              at icechunk/src/asset_manager.rs:784
  |    2: icechunk::session::commit_inner
  |            with Test one file ma

IcechunkError:   x session error: object store error service error
  | 
  | context:
  |    0: icechunk::asset_manager::update_repo_info_internal
  |            with skip_online_check=false
  |              at icechunk/src/asset_manager.rs:815
  |    1: icechunk::asset_manager::update_repo_info
  |              at icechunk/src/asset_manager.rs:784
  |    2: icechunk::session::commit_inner
  |            with Test one file max_concurrent_nodes=1 rewrite_manifests=false commit_method=NewCommit allow_empty=false
  |              at icechunk/src/session.rs:1657
  | 


In [73]:
print(repo.config.storage)

<icechunk.storage.StorageSettings>
unsafe_use_conditional_create: False
unsafe_use_conditional_update: False
unsafe_use_metadata: False
storage_class: None
metadata_storage_class: None
chunks_storage_class: None
minimum_size_for_multipart_upload: 104857600 (default)
concurrency:
    <icechunk.storage.StorageConcurrencySettings>
    max_concurrent_requests_for_object: 18 (default)
    ideal_concurrent_request_size: 12582912 (default)
retries:
    <icechunk.storage.StorageRetriesSettings>
    max_tries: 10 (default)
    initial_backoff_ms: 100 (default)
    max_backoff_ms: 180000 (default)
timeouts: None



In [80]:
import boto3
import json
import uuid
from botocore.exceptions import ClientError

with open("globcolour-source-creds.json") as f:
    source_creds = json.load(f)

s3 = boto3.client(
    "s3",
    endpoint_url=source_creds["endpoint_url"],
    region_name=source_creds["region_name"],
    aws_access_key_id=source_creds["aws_access_key_id"],
    aws_secret_access_key=source_creds["aws_secret_access_key"],
    aws_session_token=source_creds["aws_session_token"],
)

bucket = "fish-pace"
key = (
    "globcolour/"
    "test3/"
    f"_write_test_{uuid.uuid4().hex}.txt"
)

try:
    # 1. Create a new object
    response = s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=b"version 1",
    )
    print("CREATE:", response["ResponseMetadata"]["HTTPStatusCode"])

    # 2. Read it back
    response = s3.get_object(Bucket=bucket, Key=key)
    print("READ 1:", response["Body"].read())

    # 3. Overwrite the same key
    response = s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=b"version 2",
    )
    print("OVERWRITE:", response["ResponseMetadata"]["HTTPStatusCode"])

    # 4. Read it back again
    response = s3.get_object(Bucket=bucket, Key=key)
    print("READ 2:", response["Body"].read())

finally:
    # 5. Clean up
    try:
        s3.delete_object(Bucket=bucket, Key=key)
        print("DELETE: succeeded")
    except ClientError as e:
        print("DELETE failed:", e.response["Error"])

CREATE: 200
READ 1: b'version 1'
OVERWRITE: 200
READ 2: b'version 2'
DELETE: succeeded


In [5]:
import boto3
import json
import uuid
from botocore.exceptions import ClientError

with open("globcolour-source-creds.json") as f:
    source_creds = json.load(f)

s3 = boto3.client(
    "s3",
    endpoint_url=source_creds["endpoint_url"],
    region_name=source_creds["region_name"],
    aws_access_key_id=source_creds["aws_access_key_id"],
    aws_secret_access_key=source_creds["aws_secret_access_key"],
    aws_session_token=source_creds["aws_session_token"],
)

bucket = "fish-pace"
key = (
    "globcolour/"
    "test3/"
    f"_conditional_write_test_{uuid.uuid4().hex}.txt"
)

try:
    # Create object
    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=b"version 1",
    )

    # Get current ETag
    head = s3.head_object(Bucket=bucket, Key=key)
    etag = head["ETag"]
    unquoted_etag = etag.strip('"')

    # Correct conditional overwrite
    response = s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=b"version 2",
        IfMatch=etag,
    )
    print(
        "CONDITIONAL OVERWRITE:",
        response["ResponseMetadata"]["HTTPStatusCode"],
    )

    result = s3.get_object(Bucket=bucket, Key=key)
    print("READ:", result["Body"].read())
    print("Status:", response["ResponseMetadata"]["HTTPStatusCode"])
    print("ETag:", response.get("ETag"))
    print("VersionId:", response.get("VersionId"))
    print("Headers:", response["ResponseMetadata"]["HTTPHeaders"])
    
except ClientError as e:
    print("ERROR CODE:", e.response["Error"].get("Code"))
    print("MESSAGE:", e.response["Error"].get("Message"))
    print(e.response)

finally:
    try:
        s3.delete_object(Bucket=bucket, Key=key)
    except Exception:
        pass

CONDITIONAL OVERWRITE: 200
READ: b'version 2'
Status: 200
ETag: "d4ca1ed7571e2e7b1f1c375bd50fa220"
VersionId: hkgYK3ElJe.RHLjQDzZNVU89mTa792Fq
Headers: {'date': 'Fri, 24 Jul 2026 23:03:43 GMT', 'content-length': '0', 'connection': 'keep-alive', 'cf-ray': 'a2068f525f5d6c24-PDX', 'cf-cache-status': 'DYNAMIC', 'access-control-allow-origin': '*', 'etag': '"d4ca1ed7571e2e7b1f1c375bd50fa220"', 'server': 'cloudflare', 'access-control-allow-headers': '*', 'access-control-allow-methods': 'GET, HEAD, PUT, POST, DELETE, OPTIONS', 'access-control-expose-headers': '*', 'server-timing': 'total;dur=38, dispatch;dur=38, backend;dur=31, cfCacheStatus;desc="DYNAMIC", cfEdge;dur=10,cfOrigin;dur=0,cfWorker;dur=40', 'x-amz-checksum-crc32': '44ffIg==', 'x-amz-checksum-type': 'FULL_OBJECT', 'x-amz-id-2': 'UdaL6iC5CmQ0iUOFJB/TRa6qCaVjKE3IUXJJsvJF6tyrc+xnMLdNzHX2Dzi/s97QMYYPMvHukRj4um2/jZT1WQPPVskYOZ74', 'x-amz-request-id': 'PJESC2GHD069N022', 'x-amz-server-side-encryption': 'AES256', 'x-amz-version-id': 'hkgY

In [15]:
import boto3
import json
import uuid
from botocore.exceptions import ClientError

with open("globcolour-source-creds.json") as f:
    source_creds = json.load(f)

s3 = boto3.client(
    "s3",
    endpoint_url=source_creds["endpoint_url"],
    region_name=source_creds["region_name"],
    aws_access_key_id=source_creds["aws_access_key_id"],
    aws_secret_access_key=source_creds["aws_secret_access_key"],
    aws_session_token=source_creds["aws_session_token"],
)

bucket = "fish-pace"
key = (
    "globcolour/"
    "cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D/"
    "repo"
)

head = s3.head_object(
    Bucket="fish-pace",
    Key=key,
)

print(head.get("Metadata"))

{}


In [16]:
import uuid
from botocore.exceptions import ClientError

bucket = "fish-pace"
key = f"globcolour/test3/_metadata_overwrite_test_{uuid.uuid4().hex}.txt"

try:
    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=b"version 1",
    )

    etag = s3.head_object(
        Bucket=bucket,
        Key=key,
    )["ETag"].strip('"')

    response = s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=b"version 2",
        IfMatch=etag,
        Metadata={"icechunk-write-id": uuid.uuid4().hex},
    )

    print("STATUS:", response["ResponseMetadata"]["HTTPStatusCode"])
    print("ETAG:", response.get("ETag"))

    head2 = s3.head_object(Bucket=bucket, Key=key)
    print("METADATA:", head2.get("Metadata"))

finally:
    s3.delete_object(Bucket=bucket, Key=key)

STATUS: 200
ETAG: "d4ca1ed7571e2e7b1f1c375bd50fa220"
METADATA: {}


In [17]:
repo_key = key = f"globcolour/test3/repo"

before = s3.head_object(Bucket="fish-pace", Key=repo_key)

print("BEFORE")
print("ETag:", before.get("ETag"))
print("VersionId:", before.get("VersionId"))
print("Size:", before.get("ContentLength"))
print("Metadata:", before.get("Metadata"))

try:
    test_session.commit("Tiny test commit")
except Exception as e:
    print(e)

after = s3.head_object(Bucket="fish-pace", Key=repo_key)

print("AFTER")
print("ETag:", after.get("ETag"))
print("VersionId:", after.get("VersionId"))
print("Size:", after.get("ContentLength"))
print("Metadata:", after.get("Metadata"))

BEFORE
ETag: "dc3de0e0ea5ede6f99f079937d34ad8e"
VersionId: niVMI6hSSJJ5MLtwWNDnou6CmdO0pUEe
Size: 648
Metadata: {}
  x session error: object store error service error
  | 
  | context:
  |    0: icechunk::asset_manager::write_transaction_log
  |            with transaction_id=WHKBK4Y8CCQ4YBQD8Z0G
  |              at icechunk/src/asset_manager.rs:460
  |    1: icechunk::session::commit_inner
  |            with Tiny test commit max_concurrent_nodes=1 rewrite_manifests=false commit_method=NewCommit allow_empty=false
  |              at icechunk/src/session.rs:1657
  | 



  2026-07-24T23:27:20.736249Z  INFO icechunk::session: Commit started, branch_name: "main", old_snapshot_id: 1CECHNKREP0F1RSTCMT0
    at icechunk/src/session.rs:3168
    in icechunk::session::commit_inner with Tiny test commit, max_concurrent_nodes: 1, rewrite_manifests: false, commit_method: NewCommit, allow_empty: false

  2026-07-24T23:27:20.736292Z TRACE icechunk::session: Building new snapshot
    at icechunk/src/session.rs:3018
    in icechunk::session::commit_inner with Tiny test commit, max_concurrent_nodes: 1, rewrite_manifests: false, commit_method: NewCommit, allow_empty: false

  2026-07-24T23:27:20.736340Z TRACE icechunk::session: Creating transaction log, transaction_log_id: WHKBK4Y8CCQ4YBQD8Z0G
    at icechunk/src/session.rs:3099
    in icechunk::session::commit_inner with Tiny test commit, max_concurrent_nodes: 1, rewrite_manifests: false, commit_method: NewCommit, allow_empty: false

  2026-07-24T23:27:20.737105Z DEBUG icechunk::asset_manager: Writing snapshot, id: WHK

AFTER
ETag: "dc3de0e0ea5ede6f99f079937d34ad8e"
VersionId: niVMI6hSSJJ5MLtwWNDnou6CmdO0pUEe
Size: 648
Metadata: {}


In [19]:
import uuid
from botocore.exceptions import ClientError

bucket = "fish-pace"
key = f"globcolour/test3/_conditional_create_{uuid.uuid4().hex}.txt"

try:
    response = s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=b"transaction test",
        IfNoneMatch="*",
        Metadata={"icechunk-write-id": uuid.uuid4().hex},
    )

    print("STATUS:", response["ResponseMetadata"]["HTTPStatusCode"])
    print("ETAG:", response.get("ETag"))
    print("VERSION:", response.get("VersionId"))

    head = s3.head_object(Bucket=bucket, Key=key)
    print("METADATA:", head.get("Metadata"))

finally:
    try:
        s3.delete_object(Bucket=bucket, Key=key)
    except Exception:
        pass

STATUS: 200
ETAG: "eacfe0f6fed8fcff67cfb5b0fa28c894"
VERSION: wUBs3jzdjj3de.YIu0ql8UNj_10aiSmj
METADATA: {}


In [9]:
import os

os.environ["ICECHUNK_LOG"] = (
    "icechunk=trace,"
    "icechunk_arrow_object_store=trace,"
    "object_store=trace"
)
os.environ["RUST_LOG"] = ",".join([
    "icechunk=trace",
    "icechunk_s3=trace",
    "object_store=trace",
    "aws_smithy_runtime=trace",
    "aws_sdk_s3=trace",
    "aws_sigv4=trace",
    "hyper=info",
])

import icechunk as ic
import uuid
import json

with open("globcolour-source-creds.json") as f:
    source_creds = json.load(f)

test_prefix = f"globcolour/_icechunk_test_{uuid.uuid4().hex}"

print(head.get("Metadata"))

test_storage = ic.s3_storage(
    bucket="fish-pace",
    prefix=test_prefix,
    region=source_creds["region_name"],
    endpoint_url=source_creds["endpoint_url"],
    force_path_style=True,
    access_key_id=source_creds["aws_access_key_id"],
    secret_access_key=source_creds["aws_secret_access_key"],
    session_token=source_creds["aws_session_token"],
    checksum_algorithm=None,
)

test_config = ic.RepositoryConfig.default()
test_config.storage = ic.StorageSettings(
    unsafe_use_conditional_create=False,
    unsafe_use_conditional_update=False,
    unsafe_use_metadata=True,
)

test_repo = ic.Repository.create(
    storage=test_storage,
    config=test_config,
)

test_session = test_repo.writable_session("main")

from zarr.core.buffer import default_buffer_prototype

buffer = default_buffer_prototype().buffer.from_bytes(
    b'{"zarr_format":3,"node_type":"group","attributes":{}}'
)

await test_session.store.set("zarr.json", buffer)

snapshot_id = test_session.commit("Tiny test commit")

print("Commit succeeded:", snapshot_id)
print("Temporary prefix:", test_prefix)

{}


  2026-07-24T23:16:03.086693Z DEBUG icechunk::repository: Creating Repository
    at icechunk/src/repository.rs:215
    in icechunk::repository::create



IcechunkError:   x object store error service error
  | 
  | context:
  |    0: icechunk::repository::create
  |              at icechunk/src/repository.rs:207
  | 
  |-> object store error service error
  |-> service error
  |-> unhandled error (ExpiredToken)
  `-> Error { code: "ExpiredToken", message: "expired credentials" }


In [7]:
import icechunk as ic
import xarray as xr

url = "https://data.source.coop/fish-pace/globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D"
storage = ic.http_storage(url)

repo = ic.Repository.open(storage)
containers = repo.config.virtual_chunk_containers or []

store = ic.Repository.open(
    storage,
    authorize_virtual_chunk_access={
        prefix: None
        for prefix in containers
    },
).readonly_session("main").store

ds = xr.open_zarr(
    store,
    consolidated=False,
    chunks=None,
)

ds


GroupNotFoundError: No group found in store <icechunk.IcechunkStore>
read_only: True
snapshot_id: 1CECHNKREP0F1RSTCMT0
branch: None
 at path ''

In [4]:
import icechunk
config = icechunk.RepositoryConfig.default()
url = "https://data.source.coop/fish-pace/globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D"
storage = icechunk.http_storage(url)


repo = icechunk.Repository.open(
    storage,
    config=config,
    authorize_virtual_chunk_access={
        "https://s3.waw3-1.cloudferro.com/": icechunk.credentials.HttpAccess,
    },
)

history = repo.ancestry(branch="main")

for snapshot in history:
    print(snapshot)

<icechunk.snapshots.SnapshotInfo>
id: 1CECHNKREP0F1RSTCMT0
parent_id: None
written_at: datetime.datetime(2026,7,24,18,26,58,122534, tzinfo=datetime.timezone.utc)
message: Repository initialized
metadata: PySnapshotProperties({"__icechunk": JsonValue(Object {"is_root": Bool(true)})})



In [1]:
import boto3
import json

with open("globcolour-source-creds.json") as f:
    source_creds = json.load(f)

s3 = boto3.client(
    "s3",
    endpoint_url=source_creds["endpoint_url"],
    region_name=source_creds["region_name"],
    aws_access_key_id=source_creds["aws_access_key_id"],
    aws_secret_access_key=source_creds["aws_secret_access_key"],
    aws_session_token=source_creds["aws_session_token"],
)

prefix = (
    "globcolour/"
    "cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D/"
)

response = s3.list_objects_v2(
    Bucket="fish-pace",
    Prefix=prefix,
)

repo_key = f"{prefix}/repo"

head = s3.head_object(
    Bucket="fish-pace",
    Key=repo_key,
)

for obj in response.get("Contents", []):
    print(obj["Key"], obj["Size"])

globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D/chunks/29TZ29YEY8F2V4DDVE20 2400
globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D/chunks/474FCBD5GVP3G64H25Z0 2920
globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D/chunks/6G69GJA5JKE31JF739W0 2920
globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D/chunks/741SW6NWKJ5NNGA4DGT0 2400
globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D/chunks/A9BCX4CQY6S8TA9GQVTG 2920
globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D/chunks/C05FDVPJNAWRPMHBG9H0 2920
globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D/chunks/EQJCSMQT8TB5K9XTRSQG 2400
globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D/chunks/FBH9FXS7HJG6FSPFYAPG 2400
globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D/chunks/GZNCQYZNP1XMQD1WB3AG 2920
globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D/chunks/JM7XK1BG6AMN26B195JG 2400
globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D

In [12]:
import uuid
from botocore.exceptions import ClientError

bucket = "fish-pace"
key = f"globcolour/test3"

try:
    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=b"version 1",
    )

    head = s3.head_object(Bucket=bucket, Key=key)
    quoted_etag = head["ETag"]
    unquoted_etag = quoted_etag.strip('"')

    print("Quoted:  ", repr(quoted_etag))
    print("Unquoted:", repr(unquoted_etag))

    response = s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=b"version 2",
        IfMatch=unquoted_etag,
    )

    print("UNQUOTED CONDITIONAL WRITE:", response)

except ClientError as e:
    print("ERROR CODE:", e.response["Error"].get("Code"))
    print("MESSAGE:", e.response["Error"].get("Message"))
    print("STATUS:", e.response["ResponseMetadata"].get("HTTPStatusCode"))

finally:
    s3.delete_object(Bucket=bucket, Key=key)


=== REQUEST ===
PUT https://data.source.coop/fish-pace/globcolour/test3
User-Agent: Boto3/1.40.70 md/Botocore#1.40.70 ua/2.1 os/linux#6.12.64-87.122.amzn2023.x86_64 md/arch#x86_64 lang/python#3.12.12 md/pyimpl#CPython m/Z,b,N,D,e cfg/retry-mode#legacy Botocore/1.40.70
Expect: 100-continue
Transfer-Encoding: chunked
Content-Encoding: aws-chunked
X-Amz-Trailer: x-amz-checksum-crc32
X-Amz-Decoded-Content-Length: 9
x-amz-sdk-checksum-algorithm: CRC32

=== RESPONSE ===
Status: 403
Date: Fri, 24 Jul 2026 23:19:44 GMT
Content-Type: application/xml
Content-Length: 224
Connection: keep-alive
Access-Control-Allow-Origin: *
access-control-allow-headers: *
access-control-allow-methods: GET, HEAD, PUT, POST, DELETE, OPTIONS
access-control-expose-headers: *
server-timing: total;dur=0, dispatch;dur=0, cfEdge;dur=9,cfOrigin;dur=0,cfWorker;dur=1
x-request-id: a206a6c93ce2fef4
Report-To: {"group":"cf-nel","max_age":604800,"endpoints":[{"url":"https://a.nel.cloudflare.com/report/v4?s=xt%2FTULX9qAItspiI1

ClientError: An error occurred (ExpiredToken) when calling the DeleteObject operation: expired credentials

In [4]:
import json
import uuid
import hashlib
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

# ---------- CONFIG ----------
CREDS_FILE = "globcolour-source-creds.json"
BUCKET = "fish-pace"
PREFIX = f"globcolour/_icechunk_diag_{uuid.uuid4().hex}"
PREFIX = f"globcolour/test3"

# ----------------------------

def load_creds(path):
    with open(path) as f:
        return json.load(f)

def s3_client(creds):
    # path-style and retry config to mimic many proxy setups
    return boto3.client(
        "s3",
        region_name=creds["region_name"],
        endpoint_url=creds["endpoint_url"],
        aws_access_key_id=creds["aws_access_key_id"],
        aws_secret_access_key=creds["aws_secret_access_key"],
        aws_session_token=creds.get("aws_session_token"),
        config=Config(
            s3={"addressing_style": "path"},
            retries={"max_attempts": 2, "mode": "standard"},
        ),
    )

def print_ok(op, extra=None):
    print(f"[OK] {op}")
    if extra:
        print(f"     {extra}")

def print_err(op, e):
    print(f"[ERR] {op}")
    if isinstance(e, ClientError):
        r = e.response
        meta = r.get("ResponseMetadata", {})
        err = r.get("Error", {})
        print(f"     HTTPStatus: {meta.get('HTTPStatusCode')}")
        print(f"     RequestId:  {meta.get('RequestId')}")
        print(f"     HostId:     {meta.get('HostId')}")
        print(f"     Code:       {err.get('Code')}")
        print(f"     Message:    {err.get('Message')}")
    else:
        print(f"     {type(e).__name__}: {e}")

def md5_hex(b):
    return hashlib.md5(b).hexdigest()

def main():
    creds = load_creds(CREDS_FILE)
    s3 = s3_client(creds)

    key_repo = f"{PREFIX}/repo"        # key analogous to icechunk repo-info object
    key_probe = f"{PREFIX}/probe.txt"

    body1 = b'{"step":1,"msg":"hello"}'
    body2 = b'{"step":2,"msg":"overwrite"}'

    print("=== S3/Proxy diagnostic start ===")
    print(f"Bucket: {BUCKET}")
    print(f"Prefix: {PREFIX}")
    print(f"Endpoint: {creds['endpoint_url']}")
    print()

    # 0) Bucket access sanity
    try:
        s3.head_bucket(Bucket=BUCKET)
        print_ok("head_bucket")
    except Exception as e:
        print_err("head_bucket", e)
        return

    # 1) PUT new object (repo-like key)
    try:
        r = s3.put_object(Bucket=BUCKET, Key=key_repo, Body=body1, ContentType="application/json")
        print_ok("put_object create repo", f"ETag={r.get('ETag')}")
    except Exception as e:
        print_err("put_object create repo", e)
        return

    # 2) GET object
    try:
        r = s3.get_object(Bucket=BUCKET, Key=key_repo)
        data = r["Body"].read()
        print_ok("get_object repo", f"len={len(data)} md5={md5_hex(data)} ETag={r.get('ETag')}")
    except Exception as e:
        print_err("get_object repo", e)

    # 3) HEAD object
    try:
        r = s3.head_object(Bucket=BUCKET, Key=key_repo)
        print_ok(
            "head_object repo",
            f"ContentLength={r.get('ContentLength')} ETag={r.get('ETag')} LastModified={r.get('LastModified')}",
        )
    except Exception as e:
        print_err("head_object repo", e)

    # 4) OVERWRITE same key
    try:
        r = s3.put_object(Bucket=BUCKET, Key=key_repo, Body=body2, ContentType="application/json")
        print_ok("put_object overwrite repo", f"ETag={r.get('ETag')}")
    except Exception as e:
        print_err("put_object overwrite repo", e)

    # 5) GET after overwrite
    try:
        r = s3.get_object(Bucket=BUCKET, Key=key_repo)
        data = r["Body"].read()
        print_ok("get_object repo after overwrite", f"body={data!r} md5={md5_hex(data)} ETag={r.get('ETag')}")
    except Exception as e:
        print_err("get_object repo after overwrite", e)

    # 6) Conditional GET If-None-Match (expect 304 usually)
    try:
        head = s3.head_object(Bucket=BUCKET, Key=key_repo)
        etag = head.get("ETag")
        try:
            s3.get_object(Bucket=BUCKET, Key=key_repo, IfNoneMatch=etag)
            print_ok("get_object IfNoneMatch", "Proxy returned object (not 304)")
        except ClientError as ce:
            code = ce.response.get("ResponseMetadata", {}).get("HTTPStatusCode")
            if code == 304:
                print_ok("get_object IfNoneMatch", "HTTP 304 Not Modified (expected)")
            else:
                print_err("get_object IfNoneMatch", ce)
    except Exception as e:
        print_err("conditional get setup", e)

    # 7) Conditional PUT If-Match (overwrite only when etag matches)
    try:
        head = s3.head_object(Bucket=BUCKET, Key=key_repo)
        etag = head.get("ETag")
        r = s3.put_object(
            Bucket=BUCKET,
            Key=key_repo,
            Body=b'{"step":3,"msg":"if-match overwrite"}',
            ContentType="application/json",
            IfMatch=etag,   # IMPORTANT: tests CAS-like semantics
        )
        print_ok("put_object IfMatch", f"ETag={r.get('ETag')}")
    except Exception as e:
        print_err("put_object IfMatch", e)

    # 8) Conditional PUT If-None-Match=* (create-only)
    key_create_only = f"{PREFIX}/create-only.txt"
    try:
        r = s3.put_object(
            Bucket=BUCKET,
            Key=key_create_only,
            Body=b"first",
            IfNoneMatch="*",
        )
        print_ok("put_object IfNoneMatch=* create", f"ETag={r.get('ETag')}")
    except Exception as e:
        print_err("put_object IfNoneMatch=* create", e)

    # second create should fail with 412 usually
    try:
        s3.put_object(
            Bucket=BUCKET,
            Key=key_create_only,
            Body=b"second",
            IfNoneMatch="*",
        )
        print_ok("put_object IfNoneMatch=* duplicate", "Unexpectedly succeeded")
    except ClientError as ce:
        code = ce.response.get("ResponseMetadata", {}).get("HTTPStatusCode")
        if code == 412:
            print_ok("put_object IfNoneMatch=* duplicate", "HTTP 412 Precondition Failed (expected)")
        else:
            print_err("put_object IfNoneMatch=* duplicate", ce)
    except Exception as e:
        print_err("put_object IfNoneMatch=* duplicate", e)

    # 9) Separate probe key basic RW
    try:
        s3.put_object(Bucket=BUCKET, Key=key_probe, Body=b"probe")
        print_ok("put_object probe")
        data = s3.get_object(Bucket=BUCKET, Key=key_probe)["Body"].read()
        print_ok("get_object probe", f"body={data!r}")
    except Exception as e:
        print_err("probe rw", e)

    # 10) Delete cleanup
    to_delete = [{"Key": key_repo}, {"Key": key_probe}, {"Key": key_create_only}]
    try:
        r = s3.delete_objects(Bucket=BUCKET, Delete={"Objects": to_delete, "Quiet": True})
        deleted = len(r.get("Deleted", []))
        errs = r.get("Errors", [])
        print_ok("delete_objects", f"deleted={deleted} errors={len(errs)}")
        for er in errs:
            print(f"     delete error key={er.get('Key')} code={er.get('Code')} msg={er.get('Message')}")
    except Exception as e:
        print_err("delete_objects", e)

    print()
    print("=== done ===")
    print(f"Diagnostic prefix was: {PREFIX}")

if __name__ == "__main__":
    main()            

=== S3/Proxy diagnostic start ===
Bucket: fish-pace
Prefix: globcolour/test3
Endpoint: https://data.source.coop

[ERR] head_bucket
     HTTPStatus: 404
     RequestId:  
     HostId:     
     Code:       404
     Message:    Not Found


In [7]:
import json, uuid, boto3
from botocore.config import Config

with open("globcolour-source-creds.json") as f:
    c = json.load(f)

s3 = boto3.client(
    "s3",
    endpoint_url=c["endpoint_url"],
    region_name=c["region_name"],
    aws_access_key_id=c["aws_access_key_id"],
    aws_secret_access_key=c["aws_secret_access_key"],
    aws_session_token=c.get("aws_session_token"),
    config=Config(s3={"addressing_style": "path"}),
)

bucket = "fish-pace"
key = f"globcolour/test3/_trace_{uuid.uuid4().hex}.txt"

def before_sign(request, **kwargs):
    print("\n=== REQUEST ===")
    print(request.method, request.url)
    for k, v in request.headers.items():
        print(f"{k}: {v}")

def after_call(http_response, parsed, model, **kwargs):
    print("\n=== RESPONSE ===")
    print("Status:", http_response.status_code)
    for k, v in http_response.headers.items():
        print(f"{k}: {v}")

s3.meta.events.register("before-sign.s3.*", before_sign)
s3.meta.events.register("after-call.s3.*", after_call)

# create
s3.put_object(Bucket=bucket, Key=key, Body=b"v1")
h = s3.head_object(Bucket=bucket, Key=key)
etag = h["ETag"]  # quoted
# overwrite conditional
s3.put_object(Bucket=bucket, Key=key, Body=b"v2", IfMatch=etag.strip('"'))
print("\nDONE key:", key)


=== REQUEST ===
PUT https://data.source.coop/fish-pace/globcolour/test3/_trace_ee61a1b1089f455cb90b2ceeae4e8892.txt
User-Agent: Boto3/1.40.70 md/Botocore#1.40.70 ua/2.1 os/linux#6.12.64-87.122.amzn2023.x86_64 md/arch#x86_64 lang/python#3.12.12 md/pyimpl#CPython m/Z,b,N,D,e cfg/retry-mode#legacy Botocore/1.40.70
Expect: 100-continue
Transfer-Encoding: chunked
Content-Encoding: aws-chunked
X-Amz-Trailer: x-amz-checksum-crc32
X-Amz-Decoded-Content-Length: 2
x-amz-sdk-checksum-algorithm: CRC32

=== RESPONSE ===
Status: 200
Date: Fri, 24 Jul 2026 23:07:17 GMT
Content-Length: 0
Connection: keep-alive
CF-Ray: a2069488fd2a87ab-PDX
CF-Cache-Status: DYNAMIC
Access-Control-Allow-Origin: *
ETag: "6654c734ccab8f440ff0825eb443dc7f"
Server: cloudflare
access-control-allow-headers: *
access-control-allow-methods: GET, HEAD, PUT, POST, DELETE, OPTIONS
access-control-expose-headers: *
server-timing: total;dur=246, dispatch;dur=246, backend;dur=83, cfCacheStatus;desc="DYNAMIC", cfEdge;dur=10,cfOrigin;du

In [28]:
import boto3
import json
import uuid
from urllib.parse import quote
from botocore.exceptions import ClientError

with open("globcolour-source-creds.json") as f:
    source_creds = json.load(f)

s3 = boto3.client(
    "s3",
    endpoint_url=source_creds["endpoint_url"],
    region_name=source_creds["region_name"],
    aws_access_key_id=source_creds["aws_access_key_id"],
    aws_secret_access_key=source_creds["aws_secret_access_key"],
    aws_session_token=source_creds["aws_session_token"],
)

bucket = "fish-pace"

source_key = (
    "globcolour/test3/"
    f"_copy_source_{uuid.uuid4().hex}.txt"
)

destination_key = (
    "globcolour/test3/"
    f"_copy_destination_{uuid.uuid4().hex}.txt"
)

try:
    # Create source object
    s3.put_object(
        Bucket=bucket,
        Key=source_key,
        Body=b"repo version 1",
    )

    head = s3.head_object(
        Bucket=bucket,
        Key=source_key,
    )

    etag = head["ETag"]
    print("Source ETag:", etag)

    # Approximate Icechunk's conditional backup copy
    response = s3.copy_object(
        Bucket=bucket,
        Key=destination_key,
        CopySource={
            "Bucket": bucket,
            "Key": source_key,
        },
        CopySourceIfMatch=etag,
    )

    print(
        "COPY STATUS:",
        response["ResponseMetadata"]["HTTPStatusCode"],
    )
    print("COPY RESULT:", response.get("CopyObjectResult"))

    result = s3.get_object(
        Bucket=bucket,
        Key=destination_key,
    )
    print("COPIED BODY:", result["Body"].read())

except ClientError as e:
    print(
        "STATUS:",
        e.response["ResponseMetadata"].get("HTTPStatusCode"),
    )
    print("CODE:", e.response["Error"].get("Code"))
    print("MESSAGE:", e.response["Error"].get("Message"))
    print(e.response)

finally:
    for key in [source_key, destination_key]:
        try:
            s3.delete_object(Bucket=bucket, Key=key)
        except Exception:
            pass

Source ETag: "e1e763cca888e341b8510b0ba82c9b6f"
STATUS: 501
CODE: NotImplemented
MESSAGE: not implemented: server-side copy (x-amz-copy-source) is not supported
{'Error': {'Code': 'NotImplemented', 'Message': 'not implemented: server-side copy (x-amz-copy-source) is not supported', 'Resource': '/fish-pace:globcolour/test3/_copy_destination_66fc351510104a2c8fd94e9a3dcb1da8.txt'}, 'ResponseMetadata': {'HTTPStatusCode': 501, 'HTTPHeaders': {'date': 'Fri, 24 Jul 2026 23:53:29 GMT', 'content-type': 'application/xml', 'content-length': '332', 'connection': 'keep-alive', 'access-control-allow-origin': '*', 'access-control-allow-headers': '*', 'access-control-allow-methods': 'GET, HEAD, PUT, POST, DELETE, OPTIONS', 'access-control-expose-headers': '*', 'server-timing': 'total;dur=0, dispatch;dur=0, cfEdge;dur=3,cfOrigin;dur=0,cfWorker;dur=1', 'x-request-id': 'a206d83a6dce8b10', 'report-to': '{"group":"cf-nel","max_age":604800,"endpoints":[{"url":"https://a.nel.cloudflare.com/report/v4?s=DHK1yH

In [ ]:
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh
source "$HOME/.cargo/env"
cd /home/jovyan/icechunk/icechunk-python
maturin build --release

find /home/jovyan/icechunk -path "*/target/wheels/*.whl"
